In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]  = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_transformers import BeautifulSoupTransformer
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

embd = OllamaEmbeddings(model="llama3:latest", base_url="http://localhost:11434")



urls = ["https://www.goal.com/en",
        "https://www.fourfourtwo.com/",
        "https://footballinsides.com/"
    ]

# Load
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000, chunk_overlap=100
)
doc_splits = text_splitter.split_documents(docs_list)

# Add to vectorstore
vectorstore=FAISS.from_documents(
    documents=doc_splits,
    embedding=embd
)


retriever=vectorstore.as_retriever()

USER_AGENT environment variable not set, consider setting it to identify your requests.
/home/ta-seen/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
retriever.invoke("Current market news")

[Document(id='f90dce0e-756b-48b9-a169-fd388062de5d', metadata={'source': 'https://footballinsides.com/', 'title': 'Football Insides – Latest News, Tactical Analysis & More', 'description': 'Football Insides: blog with the latest news, match analysis, and tactical insights on Premier League, Champions League, and European football', 'language': 'en-US'}, page_content='August 2, 2025 \n\n\n\n\n\n\nHeung-Min Son Player Profile: From Bundesliga Talent to #1 LAFC Star \n\n\n\n\t\t\t\t\tAfter ten incredible years at Tottenham Hotspur, South Korean icon Heung-Min Son is set to make a sensational move to the Major League Soccer, joining LAFC. The transfer marks a new era for both the\t\t\t\t\n\n\n\n\n\n\nRead More\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nPlayer Profile \n\n\n\n\n\n\n\n\n \n\n\n\n\n\n\n\n\n\n \n\nJune 22, 2025 \n\n\n\n\n\n\nMohamed Salah Player Profile: From Egyptian Talent to Global Superstar \n\n\n\n\t\t\t\t\tMohamed Salah’s rise to the top of world football didn’t happen overn

### retrived finished

In [8]:
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

from pydantic import BaseModel, Field

class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""
    
    datasource: Literal["vectorstore", "websearch"] = Field(
        ...,
        description="Given a user question choose to route it to web search or a vectorstore.",
    )

llm=ChatGroq(model="qwen/qwen3-32b")
structured_llm_router = llm.with_structured_output(RouteQuery)

system = """You are an expert at routing a user question to a vectorstore or web search.
The vectorstore contains documents related to latest football news.
Use the vectorstore for questions on these topics. Otherwise, use web-search."""

route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

question_router = route_prompt | structured_llm_router

# Testing
question = "Tottenham Hotspur"
docs = retriever.invoke(question)
doc_txt = docs[1].page_content
print(question_router.invoke({"document": doc_txt, "question": question}))
print("********************************")
question = "latest football news"
docs = retriever.invoke(question)
doc_txt = docs[1].page_content
print(question_router.invoke({"document": doc_txt, "question": question}))
print("********************************")
print(
    question_router.invoke(
        {"question": "Who won the football world cup 2026 "}
    )
)


datasource='vectorstore'
********************************
datasource='vectorstore'
********************************
datasource='websearch'
